# 06 - MLP com embeddings para variaveis categoricas

**Projeto:** MPEE_2026 - Aprendizado Profundo
**Tarefa:** classificacao de familia de causa raiz (6 classes)
**Regua a superar:** HistGradientBoosting com F1-macro = 0,4135

## Contribuicao metodologica

Em vez de one-hot encoding, cada variavel categorica aprende uma representacao densa
via `nn.Embedding`. Categorias semanticamente proximas (ex.: Capacitor e Resistor) tendem
a ter vetores proximos no espaco aprendido - algo que o one-hot nao captura.

## Estrutura

| Secao | Conteudo |
|---|---|
| 0 | Configuracao (PyTorch, MPS, seed) |
| 1 | Dataset e DataLoaders |
| 2 | Arquitetura TabularEmbeddingMLP |
| 3 | Funcoes de treino e avaliacao |
| 4 | Treino config B (principal) |
| 5 | Comparacao com os baselines |
| 6 | Ablacao A vs B vs C |
| 7 | Curvas de treinamento |
| 8 | Visualizacao 2D dos embeddings |
| 9 | Matriz de confusao da MLP |
| 10 | Consolidacao final |


## Secao 0 - Configuracao

In [ ]:
from pathlib import Path
from datetime import datetime
import json, copy, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (f1_score, balanced_accuracy_score, accuracy_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.backends.mps.is_available(): torch.mps.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DADOS = RAIZ / "data" / "working"
SAIDA = RAIZ / "outputs" / "mlp"
FIG = RAIZ / "outputs" / "figuras"
SAIDA.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

CATEGORICAS = ["Test Line","Test Station","Test Group","Model Name","Duty Type","componente_grupo","turno"]
NUMERICAS = ["hora_sin","hora_cos","dia_semana","is_fim_semana"]
TARGET = "target"

print("torch:", torch.__version__, "| MPS:", torch.backends.mps.is_available(), "| device:", DEVICE)

## Secao 1 - Dataset e DataLoaders

In [ ]:
treino = pd.read_parquet(DADOS/"treino.parquet")
val    = pd.read_parquet(DADOS/"val.parquet")
teste  = pd.read_parquet(DADOS/"teste.parquet")

CLASSE_IDX = {c:i for i,c in enumerate(sorted(treino[TARGET].unique()))}
IDX_CLASSE = {i:c for c,i in CLASSE_IDX.items()}
N_CLASSES = len(CLASSE_IDX)

MAPAS = {c: {v:i for i,v in enumerate(treino[c].unique())} for c in CATEGORICAS}
DIMS_EMB = {c: min(50,(len(MAPAS[c])+1)//2) for c in CATEGORICAS}

print("Classes:", list(CLASSE_IDX.keys()))
for c in CATEGORICAS:
    print(f"  {c:22s} {len(MAPAS[c]):3d} cat -> dim {DIMS_EMB[c]}")
print("Total embedding dims:", sum(DIMS_EMB.values()))

class TabularDataset(Dataset):
    def __init__(self, df):
        self.cat = np.zeros((len(df), len(CATEGORICAS)), dtype=np.int64)
        for j,c in enumerate(CATEGORICAS):
            m = MAPAS[c]
            self.cat[:,j] = df[c].map(lambda v: m.get(v,0)).values
        self.num = df[NUMERICAS].values.astype(np.float32)
        self.y = df[TARGET].map(CLASSE_IDX).values.astype(np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self,i):
        return (torch.from_numpy(self.cat[i]), torch.from_numpy(self.num[i]),
                torch.tensor(self.y[i], dtype=torch.long))

BATCH = 64
ds_tr, ds_va, ds_te = TabularDataset(treino), TabularDataset(val), TabularDataset(teste)
dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, num_workers=0)
dl_va = DataLoader(ds_va, batch_size=BATCH*4, shuffle=False, num_workers=0)
dl_te = DataLoader(ds_te, batch_size=BATCH*4, shuffle=False, num_workers=0)

print(f"Treino: {len(ds_tr)} | Val: {len(ds_va)} | Teste: {len(ds_te)} | Batches: {len(dl_tr)}")

## Secao 2 - Arquitetura TabularEmbeddingMLP

In [ ]:
class TabularEmbeddingMLP(nn.Module):
    def __init__(self, lista_cat, dims, n_num, n_classes):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(len(MAPAS[c]), dims[c]) for c in lista_cat])
        n_in = sum(dims.values()) + n_num
        self.densa = nn.Sequential(
            nn.BatchNorm1d(n_in), nn.Dropout(0.2),
            nn.Linear(n_in, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.4),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, n_classes))
    def forward(self, xc, xn):
        e = [emb(xc[:,j]) for j,emb in enumerate(self.embs)]
        return self.densa(torch.cat(e + [xn], dim=1))

torch.manual_seed(SEED)
modelo_probe = TabularEmbeddingMLP(CATEGORICAS, DIMS_EMB, len(NUMERICAS), N_CLASSES)
print("Parametros:", f"{sum(p.numel() for p in modelo_probe.parameters()):,}")
print(modelo_probe)

## Secao 3 - Funcoes de treino e avaliacao

In [ ]:
def treinar_epoca(m, loader, otim, crit):
    m.train(); tot=cor=0; perda=0.0
    for xc,xn,y in loader:
        xc,xn,y = xc.to(DEVICE), xn.to(DEVICE), y.to(DEVICE)
        otim.zero_grad()
        logits = m(xc,xn)
        loss = crit(logits,y)
        loss.backward(); otim.step()
        perda += loss.item()*len(y)
        cor += (logits.argmax(1)==y).sum().item()
        tot += len(y)
    return perda/tot, cor/tot

@torch.no_grad()
def avaliar_loader(m, loader):
    m.eval(); P=[];Y=[];PR=[]
    for xc,xn,y in loader:
        xc,xn = xc.to(DEVICE), xn.to(DEVICE)
        lg = m(xc,xn)
        PR.append(torch.softmax(lg,dim=1).cpu().numpy())
        P.append(lg.argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y), np.concatenate(PR)

def treinar(m, dl_t, dl_v, epocas=300, patience=25, lr=1e-3, verbose=True):
    cont = treino[TARGET].map(CLASSE_IDX).value_counts().sort_index().values
    pesos = torch.tensor(1.0/cont*cont.sum()/len(cont), dtype=torch.float32).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=pesos)
    otim = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = ReduceLROnPlateau(otim, mode="max", factor=0.5, patience=10)
    hist = {"epoca":[], "loss_tr":[], "f1_tr":[], "f1_va":[], "lr":[]}
    melhor=-1; pac=0; melhor_peso=copy.deepcopy(m.state_dict())
    for ep in range(1, epocas+1):
        lt,_ = treinar_epoca(m, dl_t, otim, crit)
        p_tr,y_tr_,_ = avaliar_loader(m, dl_t)
        p_va,y_va_,_ = avaliar_loader(m, dl_v)
        f1_tr = f1_score(y_tr_, p_tr, average="macro", zero_division=0)
        f1_va = f1_score(y_va_, p_va, average="macro", zero_division=0)
        sch.step(f1_va)
        for k,v in [("epoca",ep),("loss_tr",lt),("f1_tr",f1_tr),
                    ("f1_va",f1_va),("lr",otim.param_groups[0]["lr"])]:
            hist[k].append(v)
        if f1_va > melhor:
            melhor=f1_va; pac=0; melhor_peso=copy.deepcopy(m.state_dict())
        else:
            pac += 1
            if pac >= patience:
                if verbose: print(f"  early stop ep {ep} (melhor {melhor:.4f})")
                break
        if verbose and (ep<=5 or ep%20==0):
            print(f"  ep {ep:3d} loss {lt:.4f} f1_tr {f1_tr:.4f} f1_va {f1_va:.4f}")
    m.load_state_dict(melhor_peso)
    return m, pd.DataFrame(hist), melhor

print("Funcoes definidas.")

## Secao 4 - Treino config B (principal)

In [ ]:
torch.manual_seed(SEED)
modelo_B = TabularEmbeddingMLP(CATEGORICAS, DIMS_EMB, len(NUMERICAS), N_CLASSES).to(DEVICE)
t0 = time.time()
modelo_B, hist_B, melhor_B = treinar(modelo_B, dl_tr, dl_va)
tempo_B = round(time.time()-t0, 1)

p_te, y_te_, prob_te = avaliar_loader(modelo_B, dl_te)
f1_B = f1_score(y_te_, p_te, average="macro", zero_division=0)
f1w_B = f1_score(y_te_, p_te, average="weighted", zero_division=0)
bal_B = balanced_accuracy_score(y_te_, p_te)
acc_B = accuracy_score(y_te_, p_te)

print()
print("="*70); print("CONFIG B - teste"); print("="*70)
print(f"  F1-macro    : {f1_B:.4f}   (HGB = 0.4135)")
print(f"  F1-weighted : {f1w_B:.4f}")
print(f"  Bal. acc    : {bal_B:.4f}")
print(f"  Acuracia    : {acc_B:.4f}")
print(f"  Tempo       : {tempo_B}s | epocas {len(hist_B)}")
print()
print(classification_report(y_te_, p_te, target_names=[IDX_CLASSE[i] for i in range(N_CLASSES)], zero_division=0))

torch.save(modelo_B.state_dict(), SAIDA/"06_modelo_B.pt")
hist_B.to_csv(SAIDA/"06_historico_B.csv", index=False)
with open(SAIDA/"06_metricas_B.json","w") as f:
    json.dump({"config":"B","f1_macro":float(f1_B),"f1_weighted":float(f1w_B),
               "bal_acc":float(bal_B),"acuracia":float(acc_B),
               "tempo_s":tempo_B,"epocas":len(hist_B)}, f, indent=2)
print("Salvo: 06_modelo_B.pt + historico + metricas")

## Secao 5 - Comparacao com baselines

In [ ]:
cmp = RAIZ/"outputs"/"baselines"/"07_tabela_comparativa.csv"
base = pd.read_csv(cmp)[["modelo","f1_macro_teste","bal_acc_teste"]].rename(
    columns={"f1_macro_teste":"f1_macro","bal_acc_teste":"bal_acc"}) if cmp.exists() else pd.DataFrame()
nova = pd.DataFrame([{"modelo":"MLP-Embeddings (B)","f1_macro":f1_B,"bal_acc":bal_B}])
tab = pd.concat([base,nova], ignore_index=True).sort_values("f1_macro", ascending=False)

print("="*72); print("TABELA FINAL - F1-macro teste"); print("="*72)
print(tab.to_string(index=False))
hgb_v = tab.loc[tab["modelo"]=="HistGradientBoosting","f1_macro"]
if len(hgb_v):
    d = f1_B - float(hgb_v.iloc[0])
    print(f"\nDelta MLP - HGB: {d:+.4f}")
    print("  -> MLP SUPERA" if d>0.02 else ("  -> empate tecnico" if d>-0.02 else "  -> HGB superior"))
tab.to_csv(SAIDA/"06_tabela_final.csv", index=False)
print("\nSalvo: 06_tabela_final.csv")

## Secao 6 - Ablacao A / B / C

In [ ]:
def sub_ds(df, lista):
    class DS(Dataset):
        def __init__(self, d):
            self.cat = np.zeros((len(d), len(lista)), dtype=np.int64)
            for j,c in enumerate(lista):
                m = MAPAS[c]
                self.cat[:,j] = d[c].map(lambda v: m.get(v,0)).values
            self.num = d[NUMERICAS].values.astype(np.float32)
            self.y = d[TARGET].map(CLASSE_IDX).values.astype(np.int64)
        def __len__(self): return len(self.y)
        def __getitem__(self,i):
            return (torch.from_numpy(self.cat[i]), torch.from_numpy(self.num[i]),
                    torch.tensor(self.y[i], dtype=torch.long))
    return DS(df)

def rodar(nome, lista):
    dims = {c: DIMS_EMB[c] for c in lista}
    dlt = DataLoader(sub_ds(treino, lista), batch_size=BATCH, shuffle=True)
    dlv = DataLoader(sub_ds(val, lista), batch_size=BATCH*4)
    dle = DataLoader(sub_ds(teste, lista), batch_size=BATCH*4)
    torch.manual_seed(SEED)
    m = TabularEmbeddingMLP(lista, dims, len(NUMERICAS), N_CLASSES).to(DEVICE)
    t0 = time.time()
    m,h,f1best = treinar(m, dlt, dlv, verbose=False)
    tt = round(time.time()-t0,1)
    p,y_,_ = avaliar_loader(m, dle)
    f1 = f1_score(y_, p, average="macro", zero_division=0)
    ba = balanced_accuracy_score(y_, p)
    print(f"  {nome:38s} F1={f1:.4f} bal={ba:.4f} ({tt}s, {len(h)} ep)")
    return {"config":nome, "f1_macro":f1, "bal_acc":ba, "tempo_s":tt, "epocas":len(h)}

cat_A = [c for c in CATEGORICAS if c!="componente_grupo"]
cat_C = ["componente_grupo"]
print("="*70); print("ABLACAO"); print("="*70)
resA = rodar("A: processo (sem componente)", cat_A)
print(f"  {'B: processo + componente (ja treinado)':38s} F1={f1_B:.4f} bal={bal_B:.4f}")
resB = {"config":"B: processo + componente","f1_macro":float(f1_B),"bal_acc":float(bal_B),
        "tempo_s":tempo_B,"epocas":len(hist_B)}
resC = rodar("C: somente componente", cat_C)
tab_abl = pd.DataFrame([resA,resB,resC])
print(); print(tab_abl.to_string(index=False))
print(f"\nContribuicao componente (B-A): {resB['f1_macro']-resA['f1_macro']:+.4f}")
print(f"Ganho processo sobre componente (B-C): {resB['f1_macro']-resC['f1_macro']:+.4f}")
tab_abl.to_csv(SAIDA/"06_ablacao.csv", index=False)
print("Salvo: 06_ablacao.csv")

## Secao 7 - Curvas de treinamento

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))
axes[0].plot(hist_B["epoca"], hist_B["loss_tr"], color="#2E86AB")
axes[0].set_xlabel("Epoca"); axes[0].set_ylabel("Loss"); axes[0].set_title("Loss treino"); axes[0].grid(alpha=0.3)
axes[1].plot(hist_B["epoca"], hist_B["f1_tr"], label="treino", color="#2E86AB")
axes[1].plot(hist_B["epoca"], hist_B["f1_va"], label="validacao", color="#A23B72")
axes[1].axhline(0.4135, ls="--", color="gray", label="HGB")
axes[1].set_xlabel("Epoca"); axes[1].set_ylabel("F1-macro"); axes[1].set_title("F1-macro")
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(hist_B["epoca"], hist_B["lr"], color="#F18F01")
axes[2].set_xlabel("Epoca"); axes[2].set_ylabel("LR"); axes[2].set_title("Learning rate")
axes[2].set_yscale("log"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG/"06_curvas_treinamento.png", dpi=150); plt.close()
print("Figura salva: 06_curvas_treinamento.png")

## Secao 8 - Visualizacao dos embeddings

In [ ]:
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform

idx = CATEGORICAS.index("componente_grupo")
pesos = modelo_B.embs[idx].weight.detach().cpu().numpy()
rot = [v for v,_ in sorted(MAPAS["componente_grupo"].items(), key=lambda x:x[1])]

pca = PCA(n_components=2, random_state=SEED)
proj = pca.fit_transform(pesos)

fig, ax = plt.subplots(figsize=(11,8))
ax.scatter(proj[:,0], proj[:,1], s=120, c="#2E86AB", alpha=0.7, edgecolors="k")
for i,n in enumerate(rot):
    ax.annotate(n, (proj[i,0],proj[i,1]), fontsize=8, xytext=(5,5), textcoords="offset points")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("Embeddings de componente (PCA)"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG/"06_embeddings_componente.png", dpi=150); plt.close()
print(f"Figura salva | var explicada {pca.explained_variance_ratio_.sum()*100:.1f}%")

d = squareform(pdist(pesos))
pares = [(d[i,j], rot[i], rot[j]) for i in range(len(rot)) for j in range(i+1,len(rot))]
print("\nPares mais proximos no espaco aprendido:")
for dist,a,b in sorted(pares)[:5]:
    print(f"  {a:24s} <-> {b:24s} d={dist:.3f}")

## Secao 9 - Matriz de confusao da MLP

In [ ]:
nomes = [IDX_CLASSE[i] for i in range(N_CLASSES)]
cm = confusion_matrix(y_te_, p_te, labels=list(range(N_CLASSES)))
cmn = cm.astype(float)/cm.sum(axis=1,keepdims=True)

fig, ax = plt.subplots(figsize=(8,7))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
r = [c.replace(" ","\n") for c in nomes]
ax.set_xticklabels(r, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(r, fontsize=8)
ax.set_xlabel("Predito"); ax.set_ylabel("Real"); ax.set_title("MLP - matriz de confusao")
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        v = cmn[i,j]
        if v>0.01:
            ax.text(j,i,f"{v:.2f}", ha="center", va="center",
                    color="white" if v>0.5 else "black", fontsize=8)
plt.colorbar(im, fraction=0.04); plt.tight_layout()
plt.savefig(FIG/"06_matriz_confusao_mlp.png", dpi=150); plt.close()
np.savetxt(SAIDA/"06_cm_mlp.csv", cm, delimiter=",", fmt="%d")
print("Figura salva: 06_matriz_confusao_mlp.png")

try:
    cmh = pd.read_csv(RAIZ/"outputs"/"baselines"/"08_cm_HGB.csv", header=None).values
    rh = cmh.diagonal()/cmh.sum(axis=1); rm = cm.diagonal()/cm.sum(axis=1)
    print("\nRecall por classe (HGB -> MLP):")
    for i,c in enumerate(nomes):
        print(f"  {c:30s} {rh[i]:.3f} -> {rm[i]:.3f} ({rm[i]-rh[i]:+.3f})")
except FileNotFoundError:
    print("(matriz do HGB nao encontrada)")

## Secao 10 - Consolidacao final

In [ ]:
consolidado = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "ambiente": {"torch": torch.__version__, "device": str(DEVICE), "seed": SEED},
    "config_B": {"f1_macro": float(f1_B), "f1_weighted": float(f1w_B),
                 "bal_acc": float(bal_B), "acuracia": float(acc_B),
                 "tempo_s": tempo_B, "epocas": len(hist_B)},
    "ablacao": tab_abl.to_dict(orient="records"),
    "baseline_HGB_f1_macro": 0.4135,
    "delta_vs_HGB": float(f1_B - 0.4135),
}
with open(SAIDA/"06_consolidado.json","w") as f:
    json.dump(consolidado, f, indent=2, ensure_ascii=False)

print("="*70); print("RESUMO FINAL - notebook 06"); print("="*70)
print(f"  HGB (regua)  : 0.4135")
print(f"  MLP (config B): {f1_B:.4f}")
print(f"  Delta         : {f1_B - 0.4135:+.4f}")
print()
for _, r in tab_abl.iterrows():
    print(f"  {r['config']:38s} {r['f1_macro']:.4f}")
print()
print("Arquivos em", SAIDA)
for f in sorted(SAIDA.iterdir()): print("  ", f.name)
print("\nPROXIMO PASSO: escrever o artigo.")